In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


## Imports + Settings

In [3]:
from google.colab import drive
drive.mount('/content/drive')

!pip install gensim -q

import re
import unicodedata
import random

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

import pandas as pd
import numpy as np

from nltk import word_tokenize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from gensim.models import Word2Vec

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.2 MB/s eta 0:00:00:00:0100:01


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
TOP_K_CATEGORIES = 10
SAMPLE_PER_CLASS = 2000
RANDOM_STATE = 42

CONTEXT_COLUMNS = ["L1", "L2", "L3", "L4"]

OUTER_FOLDS = 10
INNER_FOLDS = 3

HIDDEN_DIM_VALUES = [64, 128, 256]

WORD2VEC_DIM = 100
MAX_LENGTH = 80
BATCH_SIZE = 128
EPOCHS = 2
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.3

## Load Data + Build L1-L4

In [5]:
data = pd.read_json("/content/drive/MyDrive/SML/News_Category_Dataset_v3.json", lines=True)

data = data[["category", "headline", "short_description"]]

data = data.dropna()

data["category"] = data["category"].astype(str).str.strip()
data["headline"] = data["headline"].astype(str).str.strip()
data["short_description"] = data["short_description"].astype(str).str.strip()

data = data[
    (data["category"] != "") &
    (data["headline"] != "") &
    (data["short_description"] != "")
].copy()

top_10_categories = data["category"].value_counts().head(TOP_K_CATEGORIES).index.tolist()

data = data[data["category"].isin(top_10_categories)].copy()

sampled_data = []

for category in top_10_categories:
    category_data = data[data["category"] == category]
    category_sample = category_data.sample(
        n=SAMPLE_PER_CLASS,
        random_state=RANDOM_STATE
    )
    sampled_data.append(category_sample)

data = pd.concat(sampled_data)

data = data.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

category_to_label = {}

for i in range(len(top_10_categories)):
    category_to_label[top_10_categories[i]] = i

labels = []

for category in data["category"]:
    labels.append(category_to_label[category])

data["label"] = labels

In [6]:
def get_first_tokens(text, number_of_tokens):
    tokens = str(text).split()
    return " ".join(tokens[:number_of_tokens])


data["L1"] = data["headline"].apply(lambda x: get_first_tokens(x, 5))

data["L2"] = data["headline"]

data["L3"] = data["headline"] + " " + data["short_description"].apply(
    lambda x: get_first_tokens(x, 15)
)

data["L4"] = data["headline"] + " " + data["short_description"]

data = data[
    [
        "category",
        "label",
        "headline",
        "short_description",
        "L1",
        "L2",
        "L3",
        "L4"
    ]
].copy()

data.to_csv("processed_news.csv", index=False)

label_rows = []

for category in top_10_categories:
    label_rows.append({
        "category": category,
        "label": category_to_label[category]
    })

label_mapping = pd.DataFrame(label_rows)

label_mapping.to_csv("label_mapping.csv", index=False)

print("processed_news.csv saved")
print("label_mapping.csv saved")
print("Data shape:", data.shape)
print(data["category"].value_counts())
print(label_mapping)

processed_news.csv saved
label_mapping.csv saved
Data shape: (20000, 8)
category
PARENTING         2000
WELLNESS          2000
TRAVEL            2000
POLITICS          2000
FOOD & DRINK      2000
BUSINESS          2000
STYLE & BEAUTY    2000
HEALTHY LIVING    2000
ENTERTAINMENT     2000
QUEER VOICES      2000
Name: count, dtype: int64
         category  label
0        POLITICS      0
1        WELLNESS      1
2   ENTERTAINMENT      2
3  STYLE & BEAUTY      3
4          TRAVEL      4
5       PARENTING      5
6    FOOD & DRINK      6
7    QUEER VOICES      7
8  HEALTHY LIVING      8
9        BUSINESS      9


## CV & Metrics

In [7]:
def make_stratified_folds(y, number_of_folds, random_state):
    y = np.array(y)

    rng = np.random.default_rng(random_state)

    folds = []

    for i in range(number_of_folds):
        folds.append([])

    unique_labels = np.unique(y)

    for label in unique_labels:
        label_indices = np.where(y == label)[0]
        rng.shuffle(label_indices)

        split_indices = np.array_split(label_indices, number_of_folds)

        for fold_number in range(number_of_folds):
            folds[fold_number].extend(split_indices[fold_number].tolist())

    final_folds = []

    for fold in folds:
        fold = np.array(fold)
        rng.shuffle(fold)
        final_folds.append(fold)

    return final_folds


def calculate_accuracy(y_true, y_pred):
    correct_count = 0

    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct_count += 1

    return correct_count / len(y_true)


def calculate_macro_f1(y_true, y_pred):
    labels = np.unique(y_true)

    f1_scores = []

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        f1_scores.append(f1)

    return np.mean(f1_scores)


def calculate_weighted_f1(y_true, y_pred):
    labels = np.unique(y_true)

    total_count = len(y_true)
    weighted_sum = 0

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0
        support = 0

        for i in range(len(y_true)):
            if y_true[i] == label:
                support += 1

            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        weighted_sum += f1 * support

    return weighted_sum / total_count


def calculate_per_class_f1(y_true, y_pred):
    labels = np.unique(y_true)

    result = {}

    for label in labels:
        true_positive = 0
        false_positive = 0
        false_negative = 0

        for i in range(len(y_true)):
            if y_true[i] == label and y_pred[i] == label:
                true_positive += 1
            elif y_true[i] != label and y_pred[i] == label:
                false_positive += 1
            elif y_true[i] == label and y_pred[i] != label:
                false_negative += 1

        if true_positive + false_positive == 0:
            precision = 0
        else:
            precision = true_positive / (true_positive + false_positive)

        if true_positive + false_negative == 0:
            recall = 0
        else:
            recall = true_positive / (true_positive + false_negative)

        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)

        result[int(label)] = f1

    return result

## BiLSTM + Word2Vec Helper Functions

In [8]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def clean_text_for_sequence(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)

    return text


def tokenize_for_lstm(text):
    text = clean_text_for_sequence(text)
    tokens = word_tokenize(text)
    return tokens


def build_vocab_from_train_tokens(train_token_lists, min_count=1):
    word_to_index = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    word_counts = {}

    for tokens in train_token_lists:
        for token in tokens:
            if token not in word_counts:
                word_counts[token] = 0
            word_counts[token] += 1

    for word in word_counts:
        if word_counts[word] >= min_count:
            word_to_index[word] = len(word_to_index)

    return word_to_index


def tokens_to_indices(token_lists, word_to_index, max_length):
    all_indices = []

    for tokens in token_lists:
        indices = []

        for token in tokens[:max_length]:
            if token in word_to_index:
                indices.append(word_to_index[token])
            else:
                indices.append(word_to_index["<UNK>"])

        while len(indices) < max_length:
            indices.append(word_to_index["<PAD>"])

        all_indices.append(indices)

    return np.array(all_indices, dtype=np.int64)


def train_word2vec_on_train_tokens(
    train_token_lists,
    vector_size=100,
    window=5,
    min_count=1,
    seed=42
):
    word2vec_model = Word2Vec(
        sentences=train_token_lists,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=2,
        sg=1,
        seed=seed
    )

    return word2vec_model


def make_embedding_matrix(word_to_index, word2vec_model, embedding_dim):
    embedding_matrix = np.random.normal(
        loc=0,
        scale=0.05,
        size=(len(word_to_index), embedding_dim)
    )

    embedding_matrix[word_to_index["<PAD>"]] = np.zeros(embedding_dim)

    for word in word_to_index:
        index = word_to_index[word]

        if word in word2vec_model.wv:
            embedding_matrix[index] = word2vec_model.wv[word]

    return torch.tensor(embedding_matrix, dtype=torch.float32)

## Dataset + BiLSTM Model

In [9]:
class TextDataset(Dataset):
    def __init__(self, X_indices, y):
        self.X_indices = torch.tensor(X_indices, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.X_indices[index], self.y[index]

In [10]:
class BiLSTMClassifier(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        hidden_dim,
        num_classes,
        dropout_rate=0.3,
        freeze_embedding=False
    ):
        super(BiLSTMClassifier, self).__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=freeze_embedding,
            padding_idx=0
        )

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout_rate)

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)

        lstm_output, _ = self.bilstm(embedded)

        pooled = torch.mean(lstm_output, dim=1)

        pooled = self.dropout(pooled)

        logits = self.classifier(pooled)

        return logits

## Train and Predict

In [11]:
def train_bilstm_and_predict(
    X_train_text,
    y_train,
    X_test_text,
    hidden_dim,
    random_state
):
    set_all_seeds(random_state)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    train_token_lists = []

    for text in X_train_text:
        train_token_lists.append(tokenize_for_lstm(text))

    test_token_lists = []

    for text in X_test_text:
        test_token_lists.append(tokenize_for_lstm(text))


    word_to_index = build_vocab_from_train_tokens(
        train_token_lists=train_token_lists,
        min_count=1
    )

    word2vec_model = train_word2vec_on_train_tokens(
        train_token_lists=train_token_lists,
        vector_size=WORD2VEC_DIM,
        window=5,
        min_count=1,
        seed=random_state
    )

    embedding_matrix = make_embedding_matrix(
        word_to_index=word_to_index,
        word2vec_model=word2vec_model,
        embedding_dim=WORD2VEC_DIM
    )

    X_train_indices = tokens_to_indices(
        token_lists=train_token_lists,
        word_to_index=word_to_index,
        max_length=MAX_LENGTH
    )

    X_test_indices = tokens_to_indices(
        token_lists=test_token_lists,
        word_to_index=word_to_index,
        max_length=MAX_LENGTH
    )

    train_dataset = TextDataset(X_train_indices, y_train)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    num_classes = len(np.unique(data["label"]))

    model = BiLSTMClassifier(
        embedding_matrix=embedding_matrix,
        hidden_dim=hidden_dim,
        num_classes=num_classes,
        dropout_rate=DROPOUT_RATE,
        freeze_embedding=False
    )

    model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    model.train()

    for epoch in range(EPOCHS):
        total_loss = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X)

            loss = criterion(logits, batch_y)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        average_loss = total_loss / len(train_loader)
        print("Epoch:", epoch + 1, "| loss:", round(average_loss, 4))

    model.eval()

    test_dummy_labels = np.zeros(len(X_test_indices), dtype=int)

    test_dataset = TextDataset(X_test_indices, test_dummy_labels)

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    predictions = []

    with torch.no_grad():
        for batch_X, _ in test_loader:
            batch_X = batch_X.to(device)

            logits = model(batch_X)

            batch_pred = torch.argmax(logits, dim=1)

            predictions.extend(batch_pred.cpu().numpy().tolist())

    return np.array(predictions)

## Inner CV Tune Hidden Dim

In [12]:
def tune_bilstm_hidden_dim_with_inner_cv(
    X_train_text,
    y_train,
    hidden_dim_values,
    inner_folds_number,
    random_state
):
    inner_folds = make_stratified_folds(
        y_train,
        inner_folds_number,
        random_state
    )

    all_indices = np.arange(len(y_train))

    best_hidden_dim = None
    best_score = -1

    for hidden_dim in hidden_dim_values:
        fold_scores = []

        print()
        print("Trying hidden_dim =", hidden_dim)

        for inner_fold_number, valid_indices in enumerate(inner_folds, start=1):
            print("Inner fold:", inner_fold_number)

            train_indices = np.setdiff1d(all_indices, valid_indices)

            X_inner_train_text = X_train_text[train_indices]
            y_inner_train = y_train[train_indices]

            X_inner_valid_text = X_train_text[valid_indices]
            y_inner_valid = y_train[valid_indices]

            y_valid_pred = train_bilstm_and_predict(
                X_train_text=X_inner_train_text,
                y_train=y_inner_train,
                X_test_text=X_inner_valid_text,
                hidden_dim=hidden_dim,
                random_state=random_state + inner_fold_number
            )

            macro_f1 = calculate_macro_f1(y_inner_valid, y_valid_pred)

            fold_scores.append(macro_f1)

        average_score = np.mean(fold_scores)

        print("hidden_dim =", hidden_dim, "| inner macro-F1 =", round(average_score, 4))

        if average_score > best_score:
            best_score = average_score
            best_hidden_dim = hidden_dim

    return best_hidden_dim, best_score

## Outer Nested CV for One Context Level

In [13]:
def run_nested_cv_bilstm_one_setting(data, context_column, hidden_dim_values):
    X = data[context_column].astype(str).to_numpy()
    y = data["label"].to_numpy()

    outer_folds = make_stratified_folds(
        y,
        OUTER_FOLDS,
        random_state=RANDOM_STATE
    )

    all_indices = np.arange(len(y))

    fold_rows = []
    per_class_rows = []

    for outer_fold_number, test_indices in enumerate(outer_folds, start=1):
        print()
        print("====================================")
        print("Model: BiLSTM + Word2Vec")
        print("Context:", context_column)
        print("Outer fold:", outer_fold_number)
        print("====================================")

        train_indices = np.setdiff1d(all_indices, test_indices)

        X_train_text = X[train_indices]
        y_train = y[train_indices]

        X_test_text = X[test_indices]
        y_test = y[test_indices]

        best_hidden_dim, best_inner_macro_f1 = tune_bilstm_hidden_dim_with_inner_cv(
            X_train_text=X_train_text,
            y_train=y_train,
            hidden_dim_values=hidden_dim_values,
            inner_folds_number=INNER_FOLDS,
            random_state=outer_fold_number
        )

        y_test_pred = train_bilstm_and_predict(
            X_train_text=X_train_text,
            y_train=y_train,
            X_test_text=X_test_text,
            hidden_dim=best_hidden_dim,
            random_state=RANDOM_STATE + outer_fold_number
        )

        test_accuracy = calculate_accuracy(y_test, y_test_pred)
        test_macro_f1 = calculate_macro_f1(y_test, y_test_pred)
        test_weighted_f1 = calculate_weighted_f1(y_test, y_test_pred)

        print("Best hidden_dim:", best_hidden_dim)
        print("Test accuracy:", test_accuracy)
        print("Test macro-F1:", test_macro_f1)
        print("Test weighted-F1:", test_weighted_f1)

        fold_row = {
            "context_level": context_column,
            "model": "bilstm_word2vec",
            "outer_fold": outer_fold_number,
            "best_hidden_dim": best_hidden_dim,
            "inner_macro_f1": best_inner_macro_f1,
            "test_accuracy": test_accuracy,
            "test_macro_f1": test_macro_f1,
            "test_weighted_f1": test_weighted_f1
        }

        fold_rows.append(fold_row)

        per_class_f1 = calculate_per_class_f1(y_test, y_test_pred)

        for label in per_class_f1:
            per_class_rows.append({
                "context_level": context_column,
                "model": "bilstm_word2vec",
                "outer_fold": outer_fold_number,
                "label": label,
                "f1": per_class_f1[label]
            })

    fold_results = pd.DataFrame(fold_rows)
    per_class_results = pd.DataFrame(per_class_rows)

    return fold_results, per_class_results

## Run All L1-L4

In [14]:
all_bilstm_fold_results = []
all_bilstm_per_class_results = []

for context_column in CONTEXT_COLUMNS:
    fold_results, per_class_results = run_nested_cv_bilstm_one_setting(
        data=data,
        context_column=context_column,
        hidden_dim_values=HIDDEN_DIM_VALUES
    )

    all_bilstm_fold_results.append(fold_results)
    all_bilstm_per_class_results.append(per_class_results)

bilstm_fold_results_all = pd.concat(
    all_bilstm_fold_results,
    ignore_index=True
)

bilstm_per_class_results_all = pd.concat(
    all_bilstm_per_class_results,
    ignore_index=True
)

bilstm_fold_results_all.to_csv(
    "bilstm_word2vec_fold_results_all_levels.csv",
    index=False
)

bilstm_per_class_results_all.to_csv(
    "bilstm_word2vec_per_class_f1_results_all_levels.csv",
    index=False
)

print("BiLSTM fold results saved.")
print("BiLSTM per-class results saved.")


Model: BiLSTM + Word2Vec
Context: L1
Outer fold: 1

Trying hidden_dim = 64
Inner fold: 1
Device: cuda
Epoch: 1 | loss: 2.2981
Epoch: 2 | loss: 2.2143
Inner fold: 2
Device: cuda
Epoch: 1 | loss: 2.3014
Epoch: 2 | loss: 2.2104
Inner fold: 3
Device: cuda
Epoch: 1 | loss: 2.3014
Epoch: 2 | loss: 2.2173
hidden_dim = 64 | inner macro-F1 = 0.0843

Trying hidden_dim = 128
Inner fold: 1
Device: cuda
Epoch: 1 | loss: 2.3015
Epoch: 2 | loss: 2.1922
Inner fold: 2
Device: cuda
Epoch: 1 | loss: 2.3
Epoch: 2 | loss: 2.1857
Inner fold: 3
Device: cuda
Epoch: 1 | loss: 2.2947
Epoch: 2 | loss: 2.1888
hidden_dim = 128 | inner macro-F1 = 0.0929

Trying hidden_dim = 256
Inner fold: 1
Device: cuda
Epoch: 1 | loss: 2.2995
Epoch: 2 | loss: 2.2637
Inner fold: 2
Device: cuda
Epoch: 1 | loss: 2.303
Epoch: 2 | loss: 2.2262
Inner fold: 3
Device: cuda
Epoch: 1 | loss: 2.301
Epoch: 2 | loss: 2.2197
hidden_dim = 256 | inner macro-F1 = 0.0775
Device: cuda
Epoch: 1 | loss: 2.2719
Epoch: 2 | loss: 2.1149
Best hidden_dim

## Summary Results

In [15]:
summary_rows = []

for context_column in CONTEXT_COLUMNS:
    context_result = bilstm_fold_results_all[
        bilstm_fold_results_all["context_level"] == context_column
    ]

    summary_rows.append({
        "context_level": context_column,
        "model": "bilstm_word2vec",
        "mean_accuracy": context_result["test_accuracy"].mean(),
        "std_accuracy": context_result["test_accuracy"].std(),
        "mean_macro_f1": context_result["test_macro_f1"].mean(),
        "std_macro_f1": context_result["test_macro_f1"].std(),
        "mean_weighted_f1": context_result["test_weighted_f1"].mean(),
        "std_weighted_f1": context_result["test_weighted_f1"].std()
    })

bilstm_summary_results = pd.DataFrame(summary_rows)

bilstm_summary_results.to_csv(
    "bilstm_word2vec_summary_results_all_levels.csv",
    index=False
)

print("BiLSTM + Word2Vec summary results saved.")
print(bilstm_summary_results)

BiLSTM + Word2Vec summary results saved.
  context_level            model  mean_accuracy  std_accuracy  mean_macro_f1  \
0            L1  bilstm_word2vec        0.24090      0.022688       0.187719   
1            L2  bilstm_word2vec        0.35225      0.031141       0.325465   
2            L3  bilstm_word2vec        0.45445      0.031659       0.435154   
3            L4  bilstm_word2vec        0.49955      0.037321       0.488939   

   std_macro_f1  mean_weighted_f1  std_weighted_f1  
0      0.036478          0.187719         0.036478  
1      0.041078          0.325465         0.041078  
2      0.035410          0.435154         0.035410  
3      0.039323          0.488939         0.039323  
